# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

**Method:** this cell finds the project folder (on Colab, it first clones the repository given in `REPO_URL`) and defines four small helpers: one to open a weekly notebook, one to pull a saved table by its title, one to pull a saved chart, and one to check that the numbers quoted in this notebook appear in the tables they come from. It reads no warehouse data and needs no token.


In [ ]:
import base64
import json
import os
import subprocess
import sys

from IPython.display import Markdown, display

REPO_URL = "https://github.com/neilivanorencia/flyrank-ml-internship"  # the GitHub repository holding this project; used only when the notebooks are not found locally (for example on Colab)

# Locally, walk up to the project's top folder (the one holding work/notebooks); on Colab, clone the repository first.
while not os.path.isdir("work/notebooks") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")
if not os.path.isdir("work/notebooks") and "google.colab" in sys.modules and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/project"], check=True)
    os.chdir("/content/project")
assert os.path.isdir("work/notebooks"), "The weekly notebooks were not found; run this from the project folder or set REPO_URL."
NOTEBOOKS = "work/notebooks"
OUTPUTS = "work/outputs"


def notebook(name: str) -> dict:
    """Return a weekly notebook as a dictionary."""
    with open(f"{NOTEBOOKS}/{name}.ipynb", encoding="utf-8") as handle:
        return json.load(handle)


def saved_table(name: str, title: str, rows: int | None = None) -> str:
    """Return the saved Markdown table whose bold title starts with `title`, optionally cut to its first `rows` rows."""
    for cell in notebook(name)["cells"]:
        for output in cell.get("outputs", []):
            text = "".join(output.get("data", {}).get("text/markdown", []))
            if text.startswith(f"**{title}"):
                if rows is not None:
                    lines = text.splitlines()
                    start = next(i for i, line in enumerate(lines) if line.startswith(("| ---", "| :"))) + 1
                    text = "\n".join(lines[:start + rows])
                return text
    raise KeyError(f"{name}: no saved table titled '{title}'")


def saved_chart(name: str, number: int = 0) -> bytes:
    """Return the `number`-th saved chart of a weekly notebook as PNG bytes."""
    charts = [output["data"]["image/png"] for cell in notebook(name)["cells"] for output in cell.get("outputs", [])
              if "image/png" in output.get("data", {})]
    return base64.b64decode(charts[number])


def check(text: str, *values: str) -> None:
    """Fail loudly if a number quoted in this notebook is missing from the table it is quoted from."""
    missing = [value for value in values if value not in text]
    assert not missing, f"Quoted numbers not found in the source table: {missing}"


WEEKLY = {"w01_research_question": "ML-01 to ML-02", "w02_ml_task_framing": "ML-03", "w03_data_contract": "ML-04",
          "w03_feature_leakage_check": "ML-05", "w04_signal_audit": "ML-06", "w04_baseline_score": "ML-07",
          "w05_model": "ML-08", "w06_validation_audit": "ML-09", "w07_action_playbook": "ML-10"}
display(Markdown("**The weekly notebooks this capstone draws on**\n\n| Notebook | Card | Title | Findings |\n| --- | --- | --- | ---: |\n" + "\n".join(
    f"| `{name}` | {card} | {''.join(notebook(name)['cells'][0]['source']).splitlines()[0].lstrip('# ')} | "
    f"{sum(1 for cell in notebook(name)['cells'] if cell['cell_type'] == 'markdown' and ''.join(cell['source']).startswith('**Finding:**'))} |"
    for name, card in WEEKLY.items())))

## 1. Question

**The research question.** _Which existing pages are earning far fewer clicks than their position and their own site predict, beyond what bad luck explains, and will that shortfall still be there next month?_

**The decision it supports.** Every month a content editor at FlyRank has thousands of live pages and time to review a few dozen. The question is where to look first. A page that ranks well but is not clicked is a cheap opportunity: the visibility is already earned, and a better title, description, or page can turn it into traffic. But a page can also miss clicks by chance for a month and recover on its own, and reviewing it wastes the editor's time.

**Why the obvious rule fails.** A common rule flags any page with at least 500 impressions, a position in the top 20, and a click-through rate below 0.5%. On this portfolio the normal click-through rate sits between 0.24% and 0.42% at every position from 1 to 20, below the 0.5% line everywhere, so the rule flags 74% to 88% of all visible pages, including most of the pages clicking exactly as expected. Judging a page needs its own position and its own site as the yardstick, and a test for luck.

**Method:** this cell shows the evidence for the failing rule from the signal audit: the normal click-through rate at each position against the flag's cutoff.


In [ ]:
flag_curve = saved_table("w04_signal_audit", "Normal click-through by position")
display(Markdown(flag_curve))

## 2. Data

**Source.** FlyRank's internship warehouse, release `v20260703`, read with DuckDB directly from Hugging Face (`FlyRank/internship-warehouse`, gated). Four tables: the daily page performance table (78,835,655 rows, one per page per day, Search Console and GA4), the page content table (519,606 pages, keyword research and content type), the client table (104 clients, history start dates), and a 90-day query table that was used only for diagnosis. All identifiers are pseudonymous hashes; no client name, URL, or search query appears anywhere in this work.

**Unit and windows.** One row is one page in one calendar-month pair: features from month M, outcome from month M+1. Three development pairs (February→March, March→April, April→May 2026) built and tuned everything; the May→June pair stayed sealed until a single final test.

**What was excluded, and why.**

- Days with an impossible position: 163,189 March page-days showed position 0, and 101,548 showed a position between 0 and 1, mostly on busy days, which made pages look top-ranked and inflated their expected clicks. Valid positions start at 1.
- Clients whose search history starts after the first day of month M, so every client is measured over a whole month.
- Pages without impressions and a valid position in both months of a pair.
- Pages FlyRank optimized during the pair: someone already acted on them, so the next month cannot judge them.
- From the content table, every field that could postdate month M: word counts and edit dates come from a July 2026 snapshot, and optimization dates and eligibility flags record FlyRank's own later decisions. Only keyword-research fields, fixed before a page is written, are features.

**Method:** this cell shows the four labelled month pairs the work rests on.


In [ ]:
pairs_table = saved_table("w07_action_playbook", "Labelled month pairs")
check(pairs_table, "114,972", "155,707", "157,530", "137,275", "8,465", "10,031", "6,525", "2,978")
display(Markdown(pairs_table))

## 3. Methodology

**Expected clicks.** For each page and month: impressions × the pooled click-through rate of its position bin (each position from 1 to 10, then 11 to 20, 21 to 50, and beyond) × its client's factor (the site's clicks ÷ its expected clicks, +1 on both sides). The yardstick is therefore the page's own position on its own site.

**The label (a real shortfall).** Clicks in a month at most half of expected, _and_ a negative binomial test saying luck explains the gap less than 5% of the time. Clicks vary far more than a Poisson count allows (1.48 times as much for pages with 10 to 29 clicks, 41 times for pages with 1,000 or more), so the spread factor α was estimated by the method of moments on pages whose position barely moved, once per development pair (0.050, 0.032, 0.039), and the median, 0.039, was locked. The target is a real shortfall in month M+1.

**Baseline (rule B).** Missed clicks in month M (expected minus actual), counted only for pages already short by the target's rule in month M. It was chosen from three candidate rules by a criterion fixed in advance and then frozen.

**Model.** Logistic regression on 25 features: month-M traffic (impressions, clicks, click-through, position, its daily spread, days with impressions, expected clicks), rule B's two month-M signals, and keyword research (search volume, competition, cost per click, intent, content type, keyword length). A depth-3 tree, a random forest, and gradient boosting were trained the same way for comparison.

**Validation.** Walk-forward and grouped by client: each test pair is scored by models trained only on earlier pairs and only on other clients, so every test page comes from a month and a site the model never saw. A model beats rule B only if its mean Precision@50 is more than 0.02 higher, or within 0.02 and higher on Capture@50 (the share of next month's missed clicks held by the top 50). The base rate is printed beside every precision. Uncertainty comes from 1,000 client-level bootstrap resamples, and a sealed month (May→June) was scored once at the end.

**Leakage checks.** Scrambling every month M+1 column, the target, and every product field leaves all 25 features unchanged; no feature crosses the 0.90 separation alarm once page size is held fixed; a planted leak lifts the score sharply, so the harness catches leaks; and no test page appears in its model's training rows.

**Method:** this cell shows the leakage audit's summary and how much looser validation designs would have flattered the model.


In [ ]:
leakage = saved_table("w06_validation_audit", "Leakage audit (all checks passed)")
designs = saved_table("w06_validation_audit", "Means over the two test pairs, and how far each design sits above the strictest")
check(designs, "0.920", "0.890")
display(Markdown(leakage))
display(Markdown(designs))

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
